In [1]:
import kagglehub

# Download latest version
path1 = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")

print("Path to dataset files:", path1)

Using Colab cache for faster access to the 'crop-recommendation-dataset' dataset.
Path to dataset files: /kaggle/input/crop-recommendation-dataset


In [2]:
import kagglehub

# Download latest version
path2 = kagglehub.dataset_download("akshatgupta7/crop-yield-in-indian-states-dataset")

print("Path to dataset files:", path2)

Using Colab cache for faster access to the 'crop-yield-in-indian-states-dataset' dataset.
Path to dataset files: /kaggle/input/crop-yield-in-indian-states-dataset


**Prepare the Yield Data:**

Regression models predict continuous numbers (like yield amounts) rather than categories. Because this dataset contains text columns (like the state and crop names), we have to convert them into a mathematical format the model can read using a technique called One-Hot Encoding.

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib
import os

df_yield = pd.read_csv(os.path.join(path2, "crop_yield.csv"))

# 1. Select the most relevant features for predicting yield
# The Kaggle dataset contains these standard columns
features = ['Crop', 'Season', 'State', 'Annual_Rainfall', 'Fertilizer', 'Pesticide']
X = df_yield[features]
y = df_yield['Yield']

# 2. Convert text columns to numbers using One-Hot Encoding
# This creates binary columns for every state, crop, and season
X_encoded = pd.get_dummies(X, columns=['Crop', 'Season', 'State'], drop_first=True)

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")

Training features shape: (15751, 91)


**Train the Random Forest Regressor:**

While XGBoost is great, a RandomForestRegressor is highly robust against overfitting for this specific agricultural yield dataset and requires almost no hyperparameter tuning to get great results.

In [6]:
# 4. Initialize and train the Random Forest Regressor
print("Training the Yield Prediction Model...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print("Training complete!")

Training the Yield Prediction Model...
Training complete!


**Evaluate the Model:**

For regression, we do not use standard "accuracy." Instead, we use the R-squared ($R^2$) score, which tells us what percentage of the yield variance our model successfully predicts. A score above 0.90 (90%) is considered excellent.

In [7]:
# 5. Make predictions on the unseen test data
y_pred = rf_model.predict(X_test)

# 6. Calculate R-squared and Mean Absolute Error
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"\n--- Model Evaluation ---")
print(f"R-squared Score: {r2 * 100:.2f}%")
print(f"Mean Absolute Error: {mae:.2f} (This is the average error margin in yield per hectare)")


--- Model Evaluation ---
R-squared Score: 97.80%
Mean Absolute Error: 9.62 (This is the average error margin in yield per hectare)


**Test it with Custom Data:**

Let's see what the model predicts for a typical Kharif season crop in Karnataka, assuming moderate rainfall and average fertilizer use.

In [8]:
# 7. Function to format a custom input to match our One-Hot Encoded training data
def predict_yield(crop, season, state, rainfall, fertilizer, pesticide):
    # Create a dictionary with 0s for all columns
    input_dict = {col: 0 for col in X_train.columns}

    # Fill in the numerical values
    input_dict['Annual_Rainfall'] = rainfall
    input_dict['Fertilizer'] = fertilizer
    input_dict['Pesticide'] = pesticide

    # Set the specific categorical columns to True/1
    if f'Crop_{crop}' in input_dict: input_dict[f'Crop_{crop}'] = 1
    if f'Season_{season}' in input_dict: input_dict[f'Season_{season}'] = 1
    if f'State_{state}' in input_dict: input_dict[f'State_{state}'] = 1

    # Convert to DataFrame
    input_df = pd.DataFrame([input_dict])

    # Predict
    predicted_yield = rf_model.predict(input_df)
    return predicted_yield[0]

# Testing with hypothetical data
predicted = predict_yield(
    crop='Rice',
    season='Kharif',
    state='Karnataka',
    rainfall=850.5,
    fertilizer=7000,
    pesticide=250
)

print(f"\n--- Custom Yield Prediction ---")
print(f"Predicted Yield: {predicted:.2f} units per hectare")

# 8. Save the model and the expected columns for the API
joblib.dump(rf_model, 'rf_yield_model.pkl')
joblib.dump(X_train.columns, 'yield_model_columns.pkl') # We need this to format user input later
print("\nYield model saved successfully!")


--- Custom Yield Prediction ---
Predicted Yield: 2.38 units per hectare

Yield model saved successfully!


In [9]:
from google.colab import drive
import joblib
import os

# 1. Mount your Google Drive (a popup will ask for permission)
drive.mount('/content/drive')

# 2. Define the path to a dedicated folder in your Drive
drive_folder = '/content/drive/MyDrive/AgriBot_Models'

# 3. Create the folder if it doesn't already exist
os.makedirs(drive_folder, exist_ok=True)

# 4. Save the Random Forest model and the columns list directly to Drive
joblib.dump(rf_model, f'{drive_folder}/rf_yield_model.pkl')
joblib.dump(X_train.columns, f'{drive_folder}/yield_model_columns.pkl')

print(f"Yield Regression Model saved successfully to {drive_folder}!")

Mounted at /content/drive
Yield Regression Model saved successfully to /content/drive/MyDrive/AgriBot_Models!
